# Phase 7 — Publication Histories, Matched Controls

Extracts the complete publication history of every matched control.

**Input:** `data/interim/phase07_control_queue.csv`, the OpenAlex snapshot

**Output:** `data/interim/phase07_control_papers.csv`

Columns are identical to the treated extract, so the two concatenate at panel
construction. `is_focal_retraction` is always false: controls have no
retraction, and the column exists only to preserve the shared schema.

Rows are written as each batch is processed. The output runs to millions of
rows, and the works carrying them hold nested author and location structures an
order of magnitude larger than the fields retained.

In [1]:
import csv
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from snapshot import Snapshot, strip_doi, strip_id

QUEUE = "data/interim/phase07_control_queue.csv"
TREATED_PAPERS = "data/interim/phase04_papers.csv"
OUT_PAPERS = "data/interim/phase07_control_papers.csv"

PAPER_COLUMNS = [
    "author_id", "work_id", "doi", "pub_year", "type", "is_retracted_flag",
    "is_focal_retraction", "cited_by_count", "counts_by_year",
    "source_id", "source_name",
]

SCAN_COLUMNS = ["id", "doi", "publication_year", "type", "is_retracted",
                "cited_by_count", "counts_by_year", "authorships",
                "primary_location"]

# Restrict the scan for testing. None runs the full snapshot.
LIMIT_FILES = None
SKIP_FILES = 0

pd.set_option("display.width", 200)
os.makedirs("data/interim", exist_ok=True)

snap = Snapshot()
d = snap.describe("works")
print(f"works: {d['files']:,} files, {d['bytes'] / 2**30:.0f} GiB")

works: 2,446 files, 675 GiB


## The queue

In [2]:
queue = pd.read_csv(QUEUE, low_memory=False)
queue["author_id"] = queue.author_id.astype(str)
control_ids = set(queue.author_id)

print(f"controls queued  {len(control_ids):,}")
if "matched_arm" in queue.columns:
    print(f"\nby the arm each was matched to")
    print(queue.matched_arm.value_counts().to_string())
print(f"\nby pseudo-retraction year")
print(queue.first_retraction_year.value_counts().sort_index().to_string())

controls queued  48,786

by the arm each was matched to
matched_arm
AUTHOR_MISCONDUCT       25365
HONEST_ERROR            10923
EDITORIAL_COMPROMISE     7128
UNCONFIRMED_CONCERNS     4302
ETHICS_VIOLATION          723
UNCLASSIFIED              345

by pseudo-retraction year
first_retraction_year
2015     3344
2016     3416
2017     3544
2018     3860
2019     5098
2020     6221
2021     9582
2022    13721


## Extraction

In [3]:
PREFIX = "https://openalex.org/"
PREFIX_LEN = len(PREFIX)
want = pa.array(sorted(control_ids), type=pa.string())


def compact_counts(rows):
    """counts_by_year as 'YYYY:n|YYYY:n', newest first."""
    out = []
    for r in rows:
        if not r:
            out.append("")
            continue
        pairs = sorted(((c["year"], c.get("cited_by_count", 0)) for c in r
                        if c.get("year") is not None),
                       key=lambda p: p[0], reverse=True)
        out.append("|".join(f"{y}:{n}" for y, n in pairs))
    return out


class Extractor:
    """Writes matching author-work rows as each batch is processed."""

    def __init__(self, path, columns):
        self.f = open(path, "w", newline="", encoding="utf-8")
        self.writer = csv.DictWriter(self.f, fieldnames=columns)
        self.writer.writeheader()
        self.n_rows = 0
        self.authors_seen = set()

    def handle(self, tbl, path):
        auth_col = tbl.column("authorships")
        if isinstance(auth_col, pa.ChunkedArray):
            auth_col = auth_col.combine_chunks()

        parent = pc.list_parent_indices(auth_col)
        ids = auth_col.values.field("author").field("id")
        bare = pc.utf8_slice_codeunits(ids, PREFIX_LEN)

        hit = pc.fill_null(pc.is_in(bare, value_set=want), False)
        sel_ids = pc.filter(bare, hit)
        if len(sel_ids) == 0:
            return None
        sel_parent = pc.filter(parent, hit)

        w = tbl.take(sel_parent)
        loc = w.column("primary_location").to_pylist()

        rows = pd.DataFrame({
            "author_id": sel_ids.to_pylist(),
            "work_id": [strip_id(x) for x in w.column("id").to_pylist()],
            "doi": [strip_doi(x) for x in w.column("doi").to_pylist()],
            "pub_year": w.column("publication_year").to_pylist(),
            "type": w.column("type").to_pylist(),
            "is_retracted_flag": [bool(x) for x in
                                  w.column("is_retracted").to_pylist()],
            "is_focal_retraction": False,
            "cited_by_count": w.column("cited_by_count").to_pylist(),
            "counts_by_year": compact_counts(
                w.column("counts_by_year").to_pylist()),
            # primary_location.id identifies the location record; the venue is
            # one level down at primary_location.source.id.
            "source_id": [strip_id(((p or {}).get("source") or {}).get("id"))
                          if p else None for p in loc],
            "source_name": [((p or {}).get("source") or {}).get("display_name")
                            if p else None for p in loc],
        })

        self.writer.writerows(rows[PAPER_COLUMNS].to_dict("records"))
        self.f.flush()
        self.n_rows += len(rows)
        self.authors_seen.update(rows.author_id)
        return None

    def close(self):
        self.f.close()


ex = Extractor(OUT_PAPERS, PAPER_COLUMNS)
t0 = time.time()
try:
    snap.scan("works", SCAN_COLUMNS, ex.handle,
              limit_files=LIMIT_FILES, skip_files=SKIP_FILES,
              progress_every=200)
finally:
    ex.close()

print(f"\nelapsed {(time.time() - t0) / 60:.1f} min")
print(f"rows written   {ex.n_rows:,}")
print(f"controls found {len(ex.authors_seen):,} of {len(control_ids):,}")

scanning works: 2,446 files, 675.2 GiB on disk
  projecting 9 of 49 columns
  200/2,446 files | 0.0M rows | kept 0 | 21 MiB/s | eta 562m
  400/2,446 files | 46.7M rows | kept 0 | 177 MiB/s | eta 60m
  600/2,446 files | 81.3M rows | kept 0 | 177 MiB/s | eta 56m
  800/2,446 files | 119.0M rows | kept 0 | 170 MiB/s | eta 54m
  1,000/2,446 files | 155.3M rows | kept 0 | 165 MiB/s | eta 52m
  1,200/2,446 files | 192.2M rows | kept 0 | 163 MiB/s | eta 48m
  1,400/2,446 files | 230.6M rows | kept 0 | 161 MiB/s | eta 44m
  1,600/2,446 files | 271.4M rows | kept 0 | 154 MiB/s | eta 41m
  1,800/2,446 files | 326.2M rows | kept 0 | 152 MiB/s | eta 34m
  2,000/2,446 files | 380.3M rows | kept 0 | 152 MiB/s | eta 27m
  2,200/2,446 files | 434.0M rows | kept 0 | 142 MiB/s | eta 14m
  2,400/2,446 files | 493.9M rows | kept 0 | 133 MiB/s | eta 3m
  done: 510,372,821 rows scanned, 0 kept, 88.0 min

elapsed 88.0 min
rows written   6,176,685
controls found 48,786 of 48,786


## Coverage

In [4]:
missing = control_ids - ex.authors_seen
print(f"controls with no works: {len(missing):,} "
      f"({len(missing) / len(control_ids):.2%})")

if missing:
    m = queue[queue.author_id.isin(missing)]
    if "matched_arm" in m.columns:
        print(f"\nby the arm they were matched to")
        print(m.matched_arm.value_counts().to_string())
    if "works_count" in m.columns:
        print(f"\nworks_count recorded on the authors entity")
        print(m.works_count.describe().round(1).to_string())

controls with no works: 0 (0.00%)


## The extract

The column list must match the treated extract exactly. A mismatch would mean
the combined panel silently loses variables.

In [5]:
papers = pd.read_csv(OUT_PAPERS, low_memory=False)
print(f"rows            {len(papers):,}")
print(f"authors         {papers.author_id.nunique():,}")
print(f"distinct works  {papers.work_id.nunique():,}")

dupes = int(papers.duplicated(subset=["author_id", "work_id"]).sum())
if dupes:
    print(f"\n  {dupes:,} duplicate author-work rows")

if os.path.isfile(TREATED_PAPERS):
    t_cols = list(pd.read_csv(TREATED_PAPERS, nrows=1).columns)
    c_cols = list(papers.columns)
    if t_cols == c_cols:
        print(f"\ncolumns match the treated extract")
    else:
        print(f"\n  [!] column mismatch with {TREATED_PAPERS}")
        print(f"      only in treated: {[c for c in t_cols if c not in c_cols]}")
        print(f"      only in control: {[c for c in c_cols if c not in t_cols]}")

print(f"\npapers per control")
per = papers.groupby("author_id").size()
print(f"  median          {per.median():.0f}")
print(f"  mean            {per.mean():.1f}")
print(f"  90th percentile {per.quantile(0.9):.0f}")
print(f"  maximum         {per.max():,}")

sid = papers.source_id.dropna().astype(str)
print(f"\nsource_id present on {papers.source_id.notna().mean():.1%} of rows")
print(f"  distinct sources {sid.nunique():,}")
if len(sid) and not sid.str.match(r"^S\d+$").all():
    bad = sid[~sid.str.match(r"^S\d+$")].head(3).tolist()
    print(f"  [!] not all source ids are OpenAlex source identifiers: {bad}")

odd = papers[(papers.pub_year < 1900) | (papers.pub_year > 2027)]
print(f"\nrows outside 1900-2027  {len(odd):,}  "
      f"({len(odd) / len(papers):.3%})")

rows            6,176,685
authors         48,786
distinct works  5,651,613

  20,006 duplicate author-work rows

columns match the treated extract

papers per control
  median          60
  mean            126.6
  90th percentile 290
  maximum         155,356

source_id present on 88.7% of rows
  distinct sources 106,652

rows outside 1900-2027  600  (0.010%)


## Controls with retractions of their own

Phase 5 excludes authors appearing on the retracted papers in this study. It
does not exclude authors retracted elsewhere, which cannot be known before
their histories are extracted.

The OpenAlex flag disagrees with Retraction Watch on a small share of records,
so this is an upper bound.

In [6]:
flagged = papers[papers.is_retracted_flag.astype(bool)]
n_flagged = flagged.author_id.nunique()
total = papers.author_id.nunique()

print(f"controls with at least one retracted work: {n_flagged:,} "
      f"({n_flagged / total:.1%})")
print(f"retracted works among controls:            {len(flagged):,} "
      f"({len(flagged) / len(papers):.3%} of rows)")

if "matched_arm" in queue.columns:
    q = queue.set_index("author_id")
    tab = pd.DataFrame({
        "controls": q.matched_arm.value_counts(),
        "with a retraction": q.loc[
            q.index.isin(flagged.author_id.astype(str))].matched_arm
            .value_counts(),
    }).fillna(0).astype(int)
    tab["rate"] = (100 * tab["with a retraction"] / tab.controls).round(1)
    print(f"\nby the arm each control was matched to")
    print(tab.to_string())

controls with at least one retracted work: 1,717 (3.5%)
retracted works among controls:            2,808 (0.045% of rows)

by the arm each control was matched to
                      controls  with a retraction  rate
matched_arm                                            
AUTHOR_MISCONDUCT        25365                854   3.4
HONEST_ERROR             10923                403   3.7
EDITORIAL_COMPROMISE      7128                285   4.0
UNCONFIRMED_CONCERNS      4302                140   3.3
ETHICS_VIOLATION           723                 25   3.5
UNCLASSIFIED               345                 10   2.9


## Coverage of the analysis window

In [7]:
q = queue.set_index("author_id")
p = papers.dropna(subset=["pub_year"]).copy()
p["pub_year"] = p.pub_year.astype(int)
p["pseudo_year"] = p.author_id.astype(str).map(q.first_retraction_year)
p = p.dropna(subset=["pseudo_year"])
p["event_time"] = p.pub_year - p.pseudo_year.astype(int)

span = p.groupby("author_id").event_time.agg(["min", "max"])
for pre, post in [(3, 3), (6, 6)]:
    ok = int(((span["min"] <= -pre) & (span["max"] >= post)).sum())
    print(f"activity spanning -{pre} to +{post}:  {ok:,} controls "
          f"({ok / len(span):.1%})")

print(f"\nrows by event time, -8 to +8")
et = p[(p.event_time >= -8) & (p.event_time <= 8)]
print(et.event_time.value_counts().sort_index().to_string())

if "matched_arm" in q.columns:
    span2 = span.join(q[["matched_arm"]])
    ok = span2[(span2["min"] <= -6) & (span2["max"] >= 6)]
    tab = pd.DataFrame({
        "queued": q.matched_arm.value_counts(),
        "spanning": ok.matched_arm.value_counts(),
    }).fillna(0).astype(int)
    tab["rate"] = (100 * tab.spanning / tab.queued).round(1)
    print(f"\nby matched arm, activity spanning -6 to +6")
    print(tab.to_string())

activity spanning -3 to +3:  40,632 controls (83.3%)
activity spanning -6 to +6:  17,071 controls (35.0%)

rows by event time, -8 to +8
event_time
-8    199834
-7    214420
-6    234260
-5    252783
-4    277984
-3    317392
-2    338819
-1    346310
 0    327555
 1    341227
 2    337697
 3    329061
 4    263384
 5    207477
 6    157121
 7    113904
 8     71937

by matched arm, activity spanning -6 to +6
                      queued  spanning  rate
matched_arm                                 
AUTHOR_MISCONDUCT      25365      8384  33.1
HONEST_ERROR           10923      4506  41.3
EDITORIAL_COMPROMISE    7128      2335  32.8
UNCONFIRMED_CONCERNS    4302      1387  32.2
ETHICS_VIOLATION         723       302  41.8
UNCLASSIFIED             345       157  45.5
